## JIRA Cloud Full Scrape & Ingestion

Scrape resolved Technical Support tickets from **Jira Cloud** (`virginia-its.atlassian.net`).

Same as `scrape-jira-cloud.ipynb` but captures **all comments** (not just the first reply), matching `scrape-jira-1` style.

## 1. Connect to Jira Cloud

In [1]:

from jira import JIRA
from dotenv import load_dotenv
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().parent

env_path = PROJECT_ROOT / ".env"

print("Looking for .env at:", env_path)
print(".env exists:", env_path.exists())

load_dotenv(env_path)

JIRA_SERVER = os.getenv("JIRA_SERVER")
JIRA_EMAIL = os.getenv("JIRA_EMAIL")
JIRA_API_TOKEN = os.getenv("JIRA_API_TOKEN")
JIRA_PROJECT_KEY = os.getenv("JIRA_PROJECT_KEY")

print("JIRA server loaded:", bool(JIRA_SERVER))
print("Email loaded:", bool(JIRA_EMAIL))
print("API token loaded:", bool(JIRA_API_TOKEN))
print("Project key loaded:", bool(JIRA_PROJECT_KEY))

jira = JIRA(
    server=JIRA_SERVER,
    basic_auth=(JIRA_EMAIL, JIRA_API_TOKEN)
)

print("Connected to Jira.")

JQL = f'project = {JIRA_PROJECT_KEY}'

print("Project key:", JIRA_PROJECT_KEY)
print("JQL:", JQL)

batch = jira.enhanced_search_issues(
    JQL,
    maxResults=10,
    fields=["summary", "description", "comment"],
    expand="renderedFields",
)

print("Returned:", len(batch))

for issue in batch:
    print(issue.key, "-", issue.fields.summary)

projects = jira.projects()
project_keys = [p.key for p in projects]

print(f"Connected. Found {len(projects)} projects.")

if JIRA_PROJECT_KEY in project_keys:
    project = jira.project(JIRA_PROJECT_KEY)
    print(f"Project {project.key} - {project.name} found.")
else:
    print(
        f"Project {JIRA_PROJECT_KEY} not found! "
        f"Available keys: {project_keys[:20]}"
    )
    project = None

Looking for .env at: C:\Users\mayoe\OneDrive\Desktop\kb_Upload\rag-kb-upload\.env
.env exists: True
JIRA server loaded: True
Email loaded: True
API token loaded: True
Project key loaded: True


Connected to Jira.
Project key: SUP
JQL: project = SUP


Returned: 10
SUP-7697 - New Storage
SUP-7696 - Storage Request
SUP-7695 - Increase Storage
SUP-7694 - Storage Request
SUP-7693 - In case my reply didn't go through
SUP-7692 - Storage Request
SUP-7691 - What kind of Billing Information I should enter? 
SUP-7690 - Update sds-rcnode-1 reservation
SUP-7689 - Request for help on making a jupyter interface accessible to class
SUP-7688 - Jobs Stuck in CG


Connected. Found 13 projects.


Project SUP - RC Support found.


## 2. Fetch Resolved Technical Support Issues

Uses `renderedFields` so Jira returns HTML-rendered descriptions and comments.

In [2]:
import re
from html import unescape


def html_to_text(html):
    if not html:
        return ""

    text = re.sub(r'<br\s*/?>', '\n', html)
    text = re.sub(r'</p>', '\n', text)
    text = re.sub(r'</li>', '\n', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = unescape(text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()


JQL = (
    f'project = {JIRA_PROJECT_KEY} '
    'AND status = Resolved '
    'AND created >= "2026-08-12" '
    'AND created < "2026-08-18"'
)

all_issues = []
next_page_token = None

while True:
    batch = jira.enhanced_search_issues(
        JQL,
        maxResults=100,
        fields=["summary", "description", "comment"],
        expand="renderedFields",
        nextPageToken=next_page_token,
    )

    all_issues.extend(batch)

    print(f"Fetched {len(all_issues)} issues so far...")

    next_page_token = (
        batch.nextPageToken
        if hasattr(batch, "nextPageToken")
        else None
    )

    if not next_page_token:
        break

print(f"\nTotal issues fetched: {len(all_issues)}")


Fetched 9 issues so far...

Total issues fetched: 9


## 3. Analyze Description Types

In [3]:
from collections import Counter


type1 = []
type2 = []
type3 = []


for issue in all_issues:

    summary = (issue.fields.summary or "").strip()

    rendered_desc = ""

    if hasattr(issue, 'renderedFields'):
        rendered_desc = (
            getattr(
                issue.renderedFields,
                'description',
                ''
            )
            or ''
        )

    if not rendered_desc:
        raw_desc = issue.fields.description

        rendered_desc = (
            raw_desc
            if isinstance(raw_desc, str)
            else ''
        )

    desc = html_to_text(rendered_desc)

    if (
        'Description: ' in desc
        or 'Description:' in desc
    ):
        type1.append((issue, desc))

    elif (
        desc.strip().startswith('Name:')
        or 'Uid:' in desc[:200]
    ):
        type2.append((issue, desc))

    else:
        type3.append((issue, desc))


print(
    f"Type 1 — Form with Description: "
    f"{len(type1)}"
)

print(
    f"Type 2 — Pure form metadata (SKIP): "
    f"{len(type2)}"
)

print(
    f"Type 3 — Free-text: "
    f"{len(type3)}"
)

print(
    f"Total: "
    f"{len(type1) + len(type2) + len(type3)}"
)


# Decision:
# - Type 1: KEEP — extract actual content from Description: field
# - Type 2: SKIP — pure form metadata (Name/Uid/Cost-Center/...), no useful description
# - Type 3: KEEP — free-text, already useful as-is

Type 1 — Form with Description: 3
Type 2 — Pure form metadata (SKIP): 3
Type 3 — Free-text: 3
Total: 9


## 4. Process Issues

- Type 1: extract content after `Description:`, strip trailing form fields
- Type 2: skip
- Type 3: keep as-is

In [4]:
def extract_description(desc):
    """Extract actual content from form-style description."""

    parts = (
        desc.split('Description: ', 1)
        if 'Description: ' in desc
        else desc.split('Description:', 1)
    )

    content = parts[1]

    for cutoff in [
        'Classification: ',
        'Classification:'
    ]:
        if cutoff in content:
            content = content.split(cutoff)[0]

    return content.strip()


def extract_comments(issue):

    raw_comments = (
        issue.fields.comment.comments
        if hasattr(issue.fields, 'comment')
        and issue.fields.comment
        else []
    )

    rendered_comments_list = []

    if hasattr(issue, 'renderedFields'):
        rendered_comments_obj = getattr(
            issue.renderedFields,
            'comment',
            None
        )

        if rendered_comments_obj:
            rendered_comments_list = getattr(
                rendered_comments_obj,
                'comments',
                []
            )

    comments = []

    for i, raw_comment in enumerate(raw_comments):

        author = (
            raw_comment.author.displayName
            if hasattr(raw_comment, 'author')
            and raw_comment.author
            else 'Unknown'
        )

        rendered_body = ''

        if i < len(rendered_comments_list):
            rendered_body = (
                getattr(
                    rendered_comments_list[i],
                    'body',
                    ''
                )
                or ''
            )

        if not rendered_body:
            rendered_body = (
                raw_comment.body
                if isinstance(raw_comment.body, str)
                else ''
            )

        body = html_to_text(rendered_body)

        if body:
            comments.append({
                'author': author,
                'body': body
            })

    return comments


processed_issues = []
skipped = 0


for issue, desc in type1:

    description = extract_description(desc)

    if not description:
        skipped += 1
        continue

    processed_issues.append({
        'key': issue.key,
        'summary': (issue.fields.summary or "").strip(),
        'description': description,
        'comments': extract_comments(issue),
    })


for issue, desc in type2:
    skipped += 1


for issue, desc in type3:

    if not desc:
        skipped += 1
        continue

    processed_issues.append({
        'key': issue.key,
        'summary': (issue.fields.summary or "").strip(),
        'description': desc,
        'comments': extract_comments(issue),
    })


print(f"Total fetched: {len(all_issues)}")
print(
    f"Processed: {len(processed_issues)} issues "
    f"({skipped} skipped)"
)

Total fetched: 9
Processed: 4 issues (5 skipped)


## 5. PII Scrubbing

Remove names, emails, phone numbers (Presidio), and UVA computing IDs (regex).

In [5]:
import re
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

UVA_ID_RE = re.compile(r'\b[a-z]{2,4}\d{1,2}[a-z0-9]{1,3}\b', re.IGNORECASE)

def scrub(text):
    if not text:
        return text
    results = analyzer.analyze(text=text, language="en", entities=["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER"])
    text = anonymizer.anonymize(text=text, analyzer_results=results).text
    text = UVA_ID_RE.sub('<UVA_ID>', text)
    return text

for issue in processed_issues:
    issue['description'] = scrub(issue['description'])
    for comment in issue['comments']:
        comment['author'] = scrub(comment['author'])
        comment['body'] = scrub(comment['body'])

print(f"PII scrubbing complete for {len(processed_issues)} issues.")

PII scrubbing complete for 4 issues.


## 6. Preview

In [6]:
for issue in processed_issues[:3]:
    print(f"=== {issue['key']} ===")
    print(f"Summary: {issue['summary']}")
    print(f"Description: {issue['description'][:300]}...")
    print(f"Comments: {len(issue['comments'])} total")
    for j, c in enumerate(issue['comments'][:3]):
        print(f"  [{j+1}] {c['author']}: {c['body'][:120]}...")
    if len(issue['comments']) > 3:
        print(f"  ... and {len(issue['comments']) - 3} more")
    print()

=== SUP-7569 ===
Summary: HPC allocation for RC GenAI evaluation
Description: Hi, I’m an IT staff member in the Astronomy Department evaluating UVA RC’s <PERSON> service with coding-agent clients such as OpenCode and Qwen Code. This work is operational evaluation rather than part of an existing faculty research project. Am I eligible to request a small HPC allocation for this...
Comments: 5 total
  [1] <PERSON>, <PERSON> (<UVA_ID>): Hello <PERSON>,

Thanks for reaching out! Our policy requires a faculty PI (who is an owner and member of their grouper ...
  [2] <PERSON>, <PERSON> (<UVA_ID>): Hi <PERSON>, 

 Thanks for the clarification. I don’t have a faculty PI sponsoring this work because it is part of my ro...
  [3] <PERSON>, <PERSON> (<UVA_ID>): Hi <PERSON>,

We’ve added you to the hpc_training group/allocation. You can access this allocation for testing purposes ...
  ... and 2 more

=== SUP-7567 ===
Summary: Using claude science on windows
Description: I would like to use <PERSON>

## 7. Generate Markdown Files

In [7]:
from pathlib import Path
import re


# Store generated files inside this repo
OUTPUT_FOLDER = PROJECT_ROOT / "data" / "jira"

# Automatically create data/jira if it doesn't exist
OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


def safe_filename(filename):

    filename = re.sub(
        r'[<>:"/\\|?*]',
        '_',
        filename
    )

    filename = re.sub(
        r'\s+',
        ' ',
        filename
    ).strip()

    return filename[:150]


def clean_markdown_text(text):

    if not text:
        return ""

    text = text.strip()

    text = re.sub(
        r'\n{3,}',
        '\n\n',
        text
    )

    return text


print(
    f"Processed issues available: "
    f"{len(processed_issues)}"
)

print(
    f"Writing files to: "
    f"{OUTPUT_FOLDER}"
)


created = 0
failed = 0


for issue in processed_issues:

    try:

        issue_key = issue["key"]

        summary = clean_markdown_text(
            issue["summary"]
        )

        description = clean_markdown_text(
            issue["description"]
        )

        markdown = f"""# {summary}

## JIRA Issue

**Issue Key:** {issue_key}

**Source:** JIRA Technical Support Ticket

---

## User Request

{description}
"""

        if issue["comments"]:

            markdown += (
                "\n---\n\n"
                "## Resolution and Discussion\n"
            )

            for number, comment in enumerate(
                issue["comments"],
                start=1
            ):

                body = clean_markdown_text(
                    comment["body"]
                )

                if not body:
                    continue

                markdown += f"""

### Comment {number}

{body}
"""

        filename = f"jira_{issue_key}_0.md"

        file_path = (
            OUTPUT_FOLDER / filename
        )

        with open(
            file_path,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(
                markdown.strip() + "\n"
            )

        created += 1

        print(
            f"Created: {file_path.name}"
        )

    except Exception as e:

        failed += 1

        print(
            f"ERROR processing "
            f"{issue.get('key', 'unknown')}: "
            f"{e}"
        )


print("=" * 60)
print("MARKDOWN GENERATION COMPLETE")
print("=" * 60)

print(f"Created: {created} files")
print(f"Failed:  {failed} files")
print(f"Output folder: {OUTPUT_FOLDER}")

Processed issues available: 4
Writing files to: C:\Users\mayoe\OneDrive\Desktop\kb_Upload\rag-kb-upload\data\jira
Created: jira_SUP-7569_0.md
Created: jira_SUP-7567_0.md
Created: jira_SUP-7560_0.md
Created: jira_SUP-7562_0.md
MARKDOWN GENERATION COMPLETE
Created: 4 files
Failed:  0 files
Output folder: C:\Users\mayoe\OneDrive\Desktop\kb_Upload\rag-kb-upload\data\jira
